In [4]:
!pip install tensorflow==2.13.0



   ---------------------------------------- 0.0/276.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/276.6 MB 1.4 MB/s eta 0:03:23
   ---------------------------------------- 0.0/276.6 MB 1.4 MB/s eta 0:03:23
   ---------------------------------------- 0.0/276.6 MB 1.4 MB/s eta 0:03:23
   ---------------------------------------- 0.0/276.6 MB 1.4 MB/s eta 0:03:23
   ---------------------------------------- 0.0/276.6 MB 1.4 MB/s eta 0:03:23
   ---------------------------------------- 0.1/276.6 MB 218.8 kB/s eta 0:21:04
   ---------------------------------------- 0.1/276.6 MB 379.3 kB/s eta 0:12:09
   ---------------------------------------- 0.2/276.6 MB 597.1 kB/s eta 0:07:43
   ---------------------------------------- 0.2/276.6 MB 597.1 kB/s eta 0:07:43
   ---------------------------------------- 0.3/276.6 MB 656.1 kB/s eta 0:07:02
   ---------------------------------------- 0.3/276.6 MB 678.1 kB/s eta 0:06:48
   ---------------------------------------- 0.3/276.6 MB 67

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-nightly-intel 2.17.0.dev20240229 requires keras-nightly~=3.1.0.dev, which is not installed.
tables 3.8.0 requires blosc2~=2.0.0, which is not installed.
tables 3.8.0 requires cython>=0.29.21, which is not installed.
tf-nightly-intel 2.17.0.dev20240229 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.1 which is incompatible.
azure-core 1.30.0 requires typing-extensions>=4.6.0, but you have typing-extensions 4.5.0 which is incompatible.
jax 0.5.0 requires numpy>=1.25, but you have numpy 1.24.3 which is incompatible.
jaxlib 0.5.0 requires numpy>=1.25, but you have numpy 1.24.3 which is incompatible.
langchain 0.0.225 requires pydantic<2,>=1, but you have pydantic 2.10.6 which is incompatible.
pydantic 2.10.6 requires typing-extensions>=4.12.2, but you have typing-extension

   ------------------ ------------------- 137.9/276.6 MB 624.7 kB/s eta 0:03:42
   ------------------ ------------------- 138.0/276.6 MB 624.1 kB/s eta 0:03:43
   ------------------ ------------------- 138.0/276.6 MB 624.1 kB/s eta 0:03:43
   ------------------ ------------------- 138.0/276.6 MB 624.1 kB/s eta 0:03:43
   ------------------ ------------------- 138.0/276.6 MB 624.1 kB/s eta 0:03:43
   ------------------ ------------------- 138.0/276.6 MB 624.1 kB/s eta 0:03:43
   ------------------ ------------------- 138.0/276.6 MB 624.1 kB/s eta 0:03:43
   ------------------ ------------------- 138.0/276.6 MB 624.1 kB/s eta 0:03:43
   ------------------ ------------------- 138.0/276.6 MB 624.1 kB/s eta 0:03:43
   ------------------ ------------------- 138.0/276.6 MB 624.1 kB/s eta 0:03:43
   ------------------ ------------------- 138.0/276.6 MB 624.1 kB/s eta 0:03:43
   ------------------ ------------------- 138.0/276.6 MB 624.1 kB/s eta 0:03:43
   ------------------ ------------------

In [6]:
import numpy as np
import scipy.io as sio
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf

# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout



In [7]:
import tensorflow as tf
print(tf.__version__)


2.17.0-dev20240229


In [15]:
# === LOAD & PREP DATA ===

# Load Ra files
ra1 = sio.loadmat('C:\\Users\\praba\\Documents\\GitHub\\UCA-M2-SEMESTER2\\bio medical signal processing\\lab assignments\\Atrial_prabal_ghosh\\Ra1.mat')['Ra1']
ra2 = sio.loadmat('C:\\Users\\praba\\Documents\\GitHub\\UCA-M2-SEMESTER2\\bio medical signal processing\\lab assignments\\Atrial_prabal_ghosh\\Ra2.mat')['Ra2']
ra3 = sio.loadmat('C:\\Users\\praba\\Documents\\GitHub\\UCA-M2-SEMESTER2\\bio medical signal processing\\lab assignments\\Atrial_prabal_ghosh\\Ra3.mat')['Ra3']

# Merge into one big 3D array → shape: (12, 15000, 75)
Xa_all = np.concatenate([ra1, ra2, ra3], axis=2)

# Extract only V1–V6 (indices 6 to 11)
Xa_v1_v6 = Xa_all[6:12, :, :]  # shape: (6, 15000, 75)

# Rearrange to (75 patients, 15000 time steps, 6 leads)
X = np.transpose(Xa_v1_v6, (2, 1, 0))  # shape: (75, 15000, 6)


In [16]:

# === LABELS ===

# Load recurrence labels
indrecur = sio.loadmat('C:\\Users\\praba\\Documents\\GitHub\\UCA-M2-SEMESTER2\\bio medical signal processing\\lab assignments\\Atrial_prabal_ghosh\\indrecur.mat')['indrecur'].flatten() - 1
indnonrecur = sio.loadmat('C:\\Users\\praba\\Documents\\GitHub\\UCA-M2-SEMESTER2\\bio medical signal processing\\lab assignments\\Atrial_prabal_ghosh\\indnonrecur.mat')['indnonrecur'].flatten() - 1

# Create label vector: 0 = no recurrence, 1 = recurrence
y = np.zeros(75)
y[indrecur] = 1

# === SPLIT DATA ===

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [17]:
indnonrecur

array([ 0,  2,  3,  9, 13, 15, 17, 23, 26, 28, 30, 31, 33, 34, 37, 39, 42,
       43, 45, 46, 49, 50, 52, 54, 60, 62, 63, 65, 66, 72, 73],
      dtype=uint8)

In [18]:
indrecur

array([ 1,  5,  6,  7, 10, 11, 12, 14, 16, 20, 22, 25, 27, 29, 32, 36, 38,
       44, 47, 48, 53, 55, 57, 58, 61, 64, 67, 68, 69, 70, 71, 74],
      dtype=uint8)

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

# === PREP DATA FOR PYTORCH ===

# Convert numpy arrays to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# Create dataset and split into train and test
dataset = TensorDataset(X_tensor, y_tensor)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# === DEFINE CNN MODEL ===

# class CNN1D(nn.Module):
#     def __init__(self):
#         super(CNN1D, self).__init__()
#         self.conv1 = nn.Conv1d(in_channels=6, out_channels=32, kernel_size=7)
#         self.pool1 = nn.MaxPool1d(kernel_size=4)
#         self.conv2 = nn.Conv1d(32, 64, kernel_size=5)
#         self.pool2 = nn.MaxPool1d(kernel_size=4)
#         self.conv3 = nn.Conv1d(64, 128, kernel_size=3)
#         self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
#         self.fc1 = nn.Linear(128, 64)
#         self.dropout = nn.Dropout(0.5)
#         self.fc2 = nn.Linear(64, 1)
#         self.sigmoid = nn.Sigmoid()

#     def forward(self, x):
#         x = x.permute(0, 2, 1)  # shape: (batch, channels=6, timesteps=15000)
#         x = self.pool1(torch.relu(self.conv1(x)))
#         x = self.pool2(torch.relu(self.conv2(x)))
#         x = torch.relu(self.conv3(x))
#         x = self.global_avg_pool(x).squeeze(-1)  # shape: (batch, 128)
#         x = torch.relu(self.fc1(x))
#         x = self.dropout(x)
#         x = self.sigmoid(self.fc2(x)).squeeze()
#         return x
    
    
class ECGTransformer(nn.Module):
    def __init__(self, input_dim=6, seq_len=15000, patch_size=4, d_model=96, nhead=4, num_layers=2):
        super(ECGTransformer, self).__init__()
        
        self.seq_len = seq_len // patch_size
        self.patch_embed = nn.Conv1d(input_dim, d_model, kernel_size=patch_size, stride=patch_size)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)  # (B, C, T)
        x = self.patch_embed(x)  # (B, D, T//patch)
        x = x.permute(0, 2, 1)  # (B, T//patch, D)

        x = self.transformer_encoder(x)  # (B, T', D)
        x = x.permute(0, 2, 1)  # (B, D, T')
        x = self.pool(x).squeeze(-1)  # (B, D)

        return self.classifier(x).squeeze()



In [23]:

# === TRAINING SETUP ===

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = CNN1D().to(device)
model = ECGTransformer().to(device)


criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# === TRAIN THE MODEL ===

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")


RuntimeError: [enforce fail at alloc_cpu.cpp:114] data. DefaultCPUAllocator: not enough memory: you tried to allocate 1800000000 bytes.

In [21]:

# === EVALUATE ===

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        preds = (outputs > 0.5).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch_y.numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"\nTest Accuracy: {acc * 100:.2f}%")
print("\nConfusion Matrix:\n", confusion_matrix(all_labels, all_preds))
print("\nClassification Report:\n", classification_report(all_labels, all_preds))



Test Accuracy: 60.00%

Confusion Matrix:
 [[9 0]
 [6 0]]

Classification Report:
               precision    recall  f1-score   support

         0.0       0.60      1.00      0.75         9
         1.0       0.00      0.00      0.00         6

    accuracy                           0.60        15
   macro avg       0.30      0.50      0.38        15
weighted avg       0.36      0.60      0.45        15



C:\Users\praba\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\praba\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\praba\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
